# Experiment 3: source-model misspecification

Controlled shifts in the source-generating law are evaluated against policies built from a fixed ambiguity family. The experiment compares expected risk, regret, wrong-answer/abstention probabilities and evidence acquisition.


## Environment


In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import matplotlib

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)

## Model and solvers


In [ ]:

from __future__ import annotations

import itertools
from dataclasses import dataclass
from functools import lru_cache
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
from scipy.optimize import linprog


Array = np.ndarray


@dataclass(frozen=True)
class SourceSpec:
    """Nominal binary source specification."""
    alpha: float  # P(E=1 | Y=1)
    beta: float   # P(E=1 | Y=0)
    cost: float
    name: str


def make_channel(alpha: float, beta: float) -> Array:
    """
    Return a binary evidence channel with rows Y={0,1} and columns E={0,1}.
    """
    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("alpha and beta must lie in [0, 1].")
    return np.array(
        [
            [1.0 - beta, beta],
            [1.0 - alpha, alpha],
        ],
        dtype=float,
    )


def _deduplicate_channels(channels: Array, names: Sequence[str]) -> Tuple[Array, List[str]]:
    seen = set()
    kept_channels: List[Array] = []
    kept_names: List[str] = []

    for channel, name in zip(channels, names):
        key = tuple(np.round(channel.ravel(), 12))
        if key not in seen:
            seen.add(key)
            kept_channels.append(channel)
            kept_names.append(name)

    return np.stack(kept_channels), kept_names


def build_coupled_scenarios(
    sources: Sequence[SourceSpec],
    rho: float,
    eps: float = 1e-9,
) -> Tuple[Array, List[List[Tuple[float, float]]], Array, List[str]]:
    """
    Build a finite, coupled ambiguity set.

    The latent regimes shift all sources coherently. This deliberately creates
    dependence across source models so that the history-consistent and fully
    rectangular formulations need not coincide.

    Returns
    -------
    scenario_channels:
        Shape (n_scenarios, n_sources, 2, 2).
    source_projection_points:
        Per-source projected (alpha, beta) channel points for Rect-ROSE.
    nominal_channels:
        Shape (n_sources, 2, 2).
    scenario_names:
        Names of retained coupled regimes.
    """
    if rho < 0:
        raise ValueError("rho must be nonnegative.")

    regimes = [
        (0.0, 0.0, "nominal"),
        (-rho, +rho, "pessimistic"),
        (+rho, -rho, "optimistic"),
        (-rho, -rho, "low-positive-rate"),
        (+rho, +rho, "high-positive-rate"),
    ]

    raw_scenarios: List[Array] = []
    names: List[str] = []

    for delta_alpha, delta_beta, regime_name in regimes:
        scenario = []
        for source in sources:
            alpha = float(np.clip(source.alpha + delta_alpha, eps, 1.0 - eps))
            beta = float(np.clip(source.beta + delta_beta, eps, 1.0 - eps))
            scenario.append(make_channel(alpha, beta))
        raw_scenarios.append(np.stack(scenario))
        names.append(regime_name)

    scenario_channels, names = _deduplicate_channels(np.stack(raw_scenarios), names)

    source_projection_points: List[List[Tuple[float, float]]] = []
    for source_index in range(len(sources)):
        points: List[Tuple[float, float]] = []
        for scenario_index in range(scenario_channels.shape[0]):
            channel = scenario_channels[scenario_index, source_index]
            point = (float(channel[1, 1]), float(channel[0, 1]))
            if not any(np.allclose(point, existing) for existing in points):
                points.append(point)
        source_projection_points.append(points)

    nominal_channels = np.stack(
        [make_channel(source.alpha, source.beta) for source in sources]
    )

    return scenario_channels, source_projection_points, nominal_channels, names


def stop_value_precise(p_y1: float, abstention_cost: float) -> Tuple[float, str]:
    risks = [p_y1, 1.0 - p_y1, abstention_cost]
    actions = ["answer_0", "answer_1", "abstain"]
    index = int(np.argmin(risks))
    return float(risks[index]), actions[index]


def nominal_dp(
    prior_y1: float,
    channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float]], Dict]:
    """Exact nominal dynamic program for binary sources."""

    n_sources = len(costs)
    memo: Dict = {}

    def recurse(omega: Array, remaining: Tuple[int, ...]):
        key = (tuple(float(value) for value in omega), remaining)
        if key in memo:
            return memo[key]

        total = float(omega.sum())
        p_y1 = float(omega[1] / total)
        stopping_value, stopping_action = stop_value_precise(
            p_y1, abstention_cost
        )

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            for evidence in (0, 1):
                next_omega = omega * channels[source_index, :, evidence]
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    child_value = recurse(next_omega, next_remaining)[0]
                    value += evidence_probability * child_value

            continuation_values[source_index] = value

            if value < best_value - 1e-12:
                best_value = value
                best_action = f"query_{source_index}"

        memo[key] = (best_value, best_action, continuation_values)
        return memo[key]

    root = recurse(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        tuple(range(n_sources)),
    )
    return root, memo


def nominal_dp_from_state(
    omega: Array,
    channels: Array,
    costs: Array,
    abstention_cost: float,
    remaining: Tuple[int, ...],
) -> float:
    """Nominal continuation value from an unnormalized binary belief."""

    memo: Dict = {}

    def recurse(current_omega: Array, current_remaining: Tuple[int, ...]) -> float:
        key = (tuple(float(value) for value in current_omega), current_remaining)
        if key in memo:
            return memo[key]

        total = float(current_omega.sum())
        p_y1 = float(current_omega[1] / total)
        best_value, _ = stop_value_precise(p_y1, abstention_cost)

        for source_index in current_remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index
                for index in current_remaining
                if index != source_index
            )

            for evidence in (0, 1):
                next_omega = (
                    current_omega * channels[source_index, :, evidence]
                )
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    value += evidence_probability * recurse(
                        next_omega, next_remaining
                    )

            best_value = min(best_value, value)

        memo[key] = float(best_value)
        return memo[key]

    return recurse(np.asarray(omega, dtype=float), remaining)


def hcr_dp(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    History-Consistent Rectangular dynamic program over a finite scenario set.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)
        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)
        active = reach_probability > 0.0

        if not np.any(active):
            # Every complete model assigns this history zero probability.
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        posterior_y1 = likelihood[active, 1] / reach_probability[active]

        answer_0_risk = float(np.max(posterior_y1))
        answer_1_risk = float(np.max(1.0 - posterior_y1))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_values = {
                evidence: recurse(
                    history + ((source_index, evidence),),
                    next_remaining,
                )[0]
                for evidence in (0, 1)
            }

            scenario_expectations = np.zeros(n_scenarios, dtype=float)

            for evidence in (0, 1):
                numerator = (
                    likelihood
                    * scenario_channels[:, source_index, :, evidence]
                ).sum(axis=1)

                evidence_probability = np.divide(
                    numerator,
                    reach_probability,
                    out=np.zeros_like(numerator),
                    where=active,
                )

                scenario_expectations += (
                    evidence_probability * child_values[evidence]
                )

            continuation_value = float(
                costs[source_index]
                + np.max(scenario_expectations[active])
            )

            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _deduplicate_beliefs(values: Iterable[float]) -> Tuple[float, ...]:
    return tuple(sorted(set(round(float(value), 12) for value in values)))


def rectangular_dp(
    prior_y1: float,
    source_projection_points: Sequence[Sequence[Tuple[float, float]]],
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    Fully rectangular outer approximation.

    The state retains the projected belief set but discards dependence between
    the belief and the source model that generated it.
    """

    n_sources = len(costs)
    channel_sets = [
        [make_channel(alpha, beta) for alpha, beta in source_points]
        for source_points in source_projection_points
    ]

    memo: Dict = {}

    def recurse(belief_set: Tuple[float, ...], remaining: Tuple[int, ...]):
        belief_set = _deduplicate_beliefs(belief_set)
        key = (belief_set, remaining)

        if key in memo:
            return memo[key]

        beliefs = np.asarray(belief_set, dtype=float)
        if not len(beliefs):
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        answer_0_risk = float(np.max(beliefs))
        answer_1_risk = float(np.max(1.0 - beliefs))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_belief_sets: Dict[int, Tuple[float, ...]] = {}

            for evidence in (0, 1):
                posteriors: List[float] = []

                for belief in beliefs:
                    for channel in channel_sets[source_index]:
                        denominator = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )

                        if denominator > 0.0:
                            posterior = (
                                belief * channel[1, evidence] / denominator
                            )
                            posteriors.append(float(posterior))

                child_belief_sets[evidence] = _deduplicate_beliefs(posteriors)

            child_values = {
                evidence: recurse(
                    child_belief_sets[evidence], next_remaining
                )[0]
                for evidence in (0, 1)
            }

            worst_expectation = -np.inf

            for belief in beliefs:
                for channel in channel_sets[source_index]:
                    expectation = 0.0

                    for evidence in (0, 1):
                        evidence_probability = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )
                        expectation += (
                            evidence_probability * child_values[evidence]
                        )

                    worst_expectation = max(
                        worst_expectation, expectation
                    )

            continuation_value = float(
                costs[source_index] + worst_expectation
            )
            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse((prior_y1,), tuple(range(n_sources)))
    return root, memo


def static_best_response(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    adversary_distribution: Array,
):
    """
    Best deterministic policy against a distribution over fixed source models.

    The recursion returns a per-scenario risk vector. At the root, this vector
    is the expected total loss of the policy under each fixed scenario.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)

        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)

        candidates = [
            ("answer_0", likelihood[:, 1].copy(), None),
            ("answer_1", likelihood[:, 0].copy(), None),
            (
                "abstain",
                abstention_cost * reach_probability.copy(),
                None,
            ),
        ]

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )
            child_zero = recurse(
                history + ((source_index, 0),), next_remaining
            )
            child_one = recurse(
                history + ((source_index, 1),), next_remaining
            )

            risk_vector = (
                costs[source_index] * reach_probability
                + child_zero[1]
                + child_one[1]
            )

            candidates.append(
                (
                    f"query_{source_index}",
                    risk_vector,
                    (child_zero[2], child_one[2]),
                )
            )

        weighted_values = [
            float(np.dot(adversary_distribution, candidate[1]))
            for candidate in candidates
        ]

        selected_index = int(np.argmin(weighted_values))
        action, risk_vector, children = candidates[selected_index]
        policy_tree = (action, children)

        memo[key] = (
            weighted_values[selected_index],
            risk_vector,
            policy_tree,
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _solve_restricted_adversary(risk_matrix: Array):
    """
    Solve max_mu min_p <risk_p, mu> for the current policy columns.
    """

    n_policies, n_scenarios = risk_matrix.shape

    # Variables: mu_1,...,mu_S,v. Minimize -v.
    objective = np.zeros(n_scenarios + 1)
    objective[-1] = -1.0

    inequality_matrix = np.zeros((n_policies, n_scenarios + 1))
    inequality_matrix[:, :n_scenarios] = -risk_matrix
    inequality_matrix[:, -1] = 1.0
    inequality_rhs = np.zeros(n_policies)

    equality_matrix = np.zeros((1, n_scenarios + 1))
    equality_matrix[0, :n_scenarios] = 1.0
    equality_rhs = np.array([1.0])

    bounds = [(0.0, None)] * n_scenarios + [(None, None)]

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            f"Restricted static game failed: {result.message}"
        )

    adversary_distribution = result.x[:n_scenarios]
    game_value = float(result.x[-1])

    return game_value, adversary_distribution


def static_double_oracle(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    tolerance: float = 1e-9,
    max_iterations: int = 500,
):
    """
    Solve the exact finite-scenario randomized Static-ROSE game.

    A double-oracle method avoids enumerating all deterministic policy trees.
    The minimizer's best response is obtained by dynamic programming.
    """

    n_scenarios = scenario_channels.shape[0]
    adversary_distribution = np.ones(n_scenarios) / n_scenarios

    first_response, _ = static_best_response(
        prior_y1,
        scenario_channels,
        costs,
        abstention_cost,
        adversary_distribution,
    )

    policy_risk_vectors = [first_response[1]]
    policy_trees = [first_response[2]]
    observed_vectors = {tuple(np.round(first_response[1], 12))}
    convergence_history = []

    for iteration in range(max_iterations):
        risk_matrix = np.vstack(policy_risk_vectors)

        upper_bound, adversary_distribution = (
            _solve_restricted_adversary(risk_matrix)
        )

        best_response, _ = static_best_response(
            prior_y1,
            scenario_channels,
            costs,
            abstention_cost,
            adversary_distribution,
        )

        lower_bound = float(
            np.dot(adversary_distribution, best_response[1])
        )
        gap = float(upper_bound - lower_bound)

        convergence_history.append(
            {
                "iteration": iteration,
                "upper_bound": upper_bound,
                "lower_bound": lower_bound,
                "gap": gap,
                "n_policy_columns": len(policy_risk_vectors),
            }
        )

        if gap <= tolerance:
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        vector_key = tuple(np.round(best_response[1], 12))

        if vector_key in observed_vectors:
            # Numerical stalling at machine precision.
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        observed_vectors.add(vector_key)
        policy_risk_vectors.append(best_response[1])
        policy_trees.append(best_response[2])

    raise RuntimeError(
        "Static double-oracle solver reached its iteration limit."
    )


def _robust_stop_from_likelihood(
    likelihood: Array,
    abstention_cost: float,
) -> float:
    reach_probability = likelihood.sum(axis=1)
    active = reach_probability > 0.0

    if not np.any(active):
        return 0.0

    posterior_y1 = (
        likelihood[active, 1] / reach_probability[active]
    )

    return float(
        min(
            np.max(posterior_y1),
            np.max(1.0 - posterior_y1),
            abstention_cost,
        )
    )


def root_acquisition_certificates(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
):
    """
    Compute root-level perfect-information lower bounds and query-then-stop
    upper bounds for all source actions.
    """

    import pandas as pd

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    root_likelihood = np.tile(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        (n_scenarios, 1),
    )

    hcr_root, _ = hcr_dp(
        prior_y1, scenario_channels, costs, abstention_cost
    )
    stopping_value = float(min(hcr_root[3]))

    rows = []

    for source_index in range(n_sources):
        remaining = tuple(
            index for index in range(n_sources) if index != source_index
        )

        child_stopping_values = {}
        for evidence in (0, 1):
            next_likelihood = (
                root_likelihood
                * scenario_channels[:, source_index, :, evidence]
            )
            child_stopping_values[evidence] = (
                _robust_stop_from_likelihood(
                    next_likelihood, abstention_cost
                )
            )

        lower_scenario_values = []
        upper_scenario_values = []

        for scenario_index in range(n_scenarios):
            perfect_information_expectation = 0.0
            query_then_stop_expectation = 0.0

            for evidence in (0, 1):
                next_omega = (
                    root_likelihood[scenario_index]
                    * scenario_channels[
                        scenario_index, source_index, :, evidence
                    ]
                )
                evidence_probability = float(next_omega.sum())

                if evidence_probability > 0.0:
                    perfect_information_expectation += (
                        evidence_probability
                        * nominal_dp_from_state(
                            next_omega,
                            scenario_channels[scenario_index],
                            costs,
                            abstention_cost,
                            remaining,
                        )
                    )
                    query_then_stop_expectation += (
                        evidence_probability
                        * child_stopping_values[evidence]
                    )

            lower_scenario_values.append(
                costs[source_index] + perfect_information_expectation
            )
            upper_scenario_values.append(
                costs[source_index] + query_then_stop_expectation
            )

        lower_bound = float(max(lower_scenario_values))
        upper_bound = float(max(upper_scenario_values))
        exact_hcr_value = float(
            hcr_root[2][source_index]
        )

        rows.append(
            {
                "source_index": source_index,
                "LB": lower_bound,
                "C_HCR": exact_hcr_value,
                "UB": upper_bound,
                "stop_value": stopping_value,
                "bound_valid": (
                    lower_bound
                    <= exact_hcr_value + 1e-10
                    and exact_hcr_value
                    <= upper_bound + 1e-10
                ),
                "rejection_certificate": (
                    lower_bound >= stopping_value - 1e-12
                ),
                "no_harm_certificate": (
                    upper_bound < stopping_value - 1e-12
                ),
            }
        )

    return pd.DataFrame(rows)


def posterior_dilation_unit_test() -> Dict[str, float]:
    """
    Reproduce the analytical cost-free retrieval dilation example.
    """

    prior_y1 = 0.9
    abstention_cost = 0.4
    costs = np.array([0.0])

    channel_a = make_channel(alpha=0.9, beta=0.1)
    channel_b = make_channel(alpha=0.1, beta=0.9)

    scenarios = np.stack(
        [
            np.stack([channel_a]),
            np.stack([channel_b]),
        ]
    )

    root, _ = hcr_dp(
        prior_y1, scenarios, costs, abstention_cost
    )

    initial_stopping_value = float(min(root[3]))
    continuation_value = float(root[2][0])
    robust_value_of_evidence = (
        initial_stopping_value - continuation_value
    )

    expected = {
        "initial_stopping_value": 0.1,
        "post_retrieval_hcr_value": 0.4,
        "RVoE_HCR": -0.3,
    }

    actual = {
        "initial_stopping_value": initial_stopping_value,
        "post_retrieval_hcr_value": continuation_value,
        "RVoE_HCR": robust_value_of_evidence,
    }

    for key, expected_value in expected.items():
        if not np.isclose(actual[key], expected_value, atol=1e-10):
            raise AssertionError(
                f"Dilation unit test failed for {key}: "
                f"expected {expected_value}, obtained {actual[key]}."
            )

    return actual




## Configuration and experiment functions

Diagnostic and final modes have separate sample counts. The stored default is final. Checkpoints support resumption with the same settings.


In [ ]:

# Experiment 3: Source-Model Misspecification Stress Test


from pathlib import Path
from typing import Any, Callable
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linprog



# Configuration


SEED = 20260714

# Begin with "diagnostic". Change to "final" only after the diagnostic
# execution finishes successfully.
RUN_MODE = "final"  # "diagnostic" or "final"

if RUN_MODE == "diagnostic":
    N_INSTANCES = 20
    SHIFT_LEVELS = [0.0, 0.30, 1.00]
    CHECKPOINT_EVERY = 2
elif RUN_MODE == "final":
    N_INSTANCES = 500
    SHIFT_LEVELS = [
        0.00,
        0.05,
        0.10,
        0.20,
        0.30,
        0.50,
        0.75,
        1.00,
    ]
    CHECKPOINT_EVERY = 10
else:
    raise ValueError("RUN_MODE must be 'diagnostic' or 'final'.")

SHIFT_TYPES = [
    "degradation",
    "asymmetric_bias",
    "adversarial_source",
    "common_mode_failure",
]

METHODS = [
    "InternalOnly",
    "RetrieveAll",
    "NominalSequential",
    "StaticROSE",
    "HCRROSE",
    "RectROSE",
    "Oracle",
]

N_SOURCES = 3

# Fixed ambiguity family used to construct robust policies.
ROBUST_LEVELS = [0.05, 0.10, 0.20]

# Fraction of sources attacked in the adversarial-source shift.
ADVERSARIAL_SOURCE_FRACTION = 1.0 / 3.0

# Numerical tolerances.
STATIC_SOLVER_TOL = 1e-9
RISK_TOL = 1e-10

# Resume an interrupted execution.
RESUME = True

# Primary local project path. On Vast.ai or another machine, the code
# falls back to the notebook's current directory.
PROJECT_DIR = Path("/Users/arian/ROSE-RAG")
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = (
    PROJECT_DIR
    / f"rose_rag_misspecification_{RUN_MODE}_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("N_INSTANCES:", N_INSTANCES)
print("SHIFT_LEVELS:", SHIFT_LEVELS)
print("SHIFT_TYPES:", SHIFT_TYPES)
print("OUTPUT_DIR:", OUTPUT_DIR)


def save_vector_figure(figure, stem: str) -> None:
    """Save only vector PDF and SVG figures."""
    figure.savefig(
        OUTPUT_DIR / f"{stem}.pdf",
        format="pdf",
        bbox_inches="tight",
    )
    figure.savefig(
        OUTPUT_DIR / f"{stem}.svg",
        format="svg",
        bbox_inches="tight",
    )



# METRIC UTILITIES


METRIC_KEYS = [
    "total_risk",
    "wrong_answer_probability",
    "abstention_probability",
    "answer_probability",
    "acquisition_cost",
    "expected_query_count",
]


def empty_metrics() -> dict[str, float]:
    return {key: 0.0 for key in METRIC_KEYS}


def terminal_metrics(
    action: str,
    true_p_y1: float,
    abstention_cost: float,
) -> dict[str, float]:
    result = empty_metrics()

    if action == "answer_0":
        result["wrong_answer_probability"] = true_p_y1
        result["answer_probability"] = 1.0

    elif action == "answer_1":
        result["wrong_answer_probability"] = 1.0 - true_p_y1
        result["answer_probability"] = 1.0

    elif action == "abstain":
        result["abstention_probability"] = 1.0

    else:
        raise ValueError(f"Unknown terminal action: {action}")

    result["total_risk"] = (
        result["wrong_answer_probability"]
        + abstention_cost
        * result["abstention_probability"]
    )
    return result


def weighted_metric_sum(
    weighted_items: list[
        tuple[float, dict[str, float]]
    ],
) -> dict[str, float]:
    result = empty_metrics()

    for weight, metrics in weighted_items:
        for key in METRIC_KEYS:
            result[key] += float(weight) * float(metrics[key])

    return result


def add_query_to_metrics(
    metrics: dict[str, float],
    source_cost: float,
) -> dict[str, float]:
    result = dict(metrics)
    result["acquisition_cost"] += float(source_cost)
    result["expected_query_count"] += 1.0
    result["total_risk"] += float(source_cost)
    return result


def combine_true_regimes(
    regime_metrics: list[dict[str, float]],
    regime_weights: np.ndarray,
) -> dict[str, float]:
    return weighted_metric_sum(
        [
            (float(weight), metrics)
            for weight, metrics in zip(
                regime_weights,
                regime_metrics,
            )
        ]
    )


def root_action_probabilities(
    actions: list[str],
    weights: np.ndarray,
) -> dict[str, float]:
    result = {
        "root_answer_probability": 0.0,
        "root_query_probability": 0.0,
        "root_abstain_probability": 0.0,
    }

    for action, weight in zip(actions, weights):
        if action.startswith("query_"):
            result["root_query_probability"] += float(weight)
        elif action == "abstain":
            result["root_abstain_probability"] += float(weight)
        else:
            result["root_answer_probability"] += float(weight)

    return result



# RANDOM NOMINAL PROBLEMS


def generate_nominal_problem(
    instance_id: int,
) -> dict[str, Any]:
    rng = np.random.default_rng(
        np.random.SeedSequence(
            [SEED, instance_id]
        )
    )

    sources: list[SourceSpec] = []

    for source_index in range(N_SOURCES):
        while True:
            alpha = float(rng.uniform(0.72, 0.93))
            beta = float(rng.uniform(0.07, 0.28))

            if alpha - beta >= 0.38:
                break

        cost = float(rng.uniform(0.008, 0.08))

        sources.append(
            SourceSpec(
                alpha=alpha,
                beta=beta,
                cost=cost,
                name=f"S{source_index + 1}",
            )
        )

    prior_y1 = float(rng.uniform(0.10, 0.90))
    abstention_cost = float(rng.uniform(0.20, 0.45))

    nominal_channels = np.stack(
        [
            make_channel(
                source.alpha,
                source.beta,
            )
            for source in sources
        ]
    )

    costs = np.array(
        [source.cost for source in sources],
        dtype=float,
    )

    return {
        "sources": sources,
        "prior_y1": prior_y1,
        "abstention_cost": abstention_cost,
        "nominal_channels": nominal_channels,
        "costs": costs,
    }



# ROBUST AMBIGUITY FAMILY


def build_robust_scenario_family(
    sources: list[SourceSpec],
) -> tuple[
    np.ndarray,
    list[list[tuple[float, float]]],
    list[str],
]:
    """
    Construct a finite coupled ambiguity family around the nominal model.

    It contains:
    - coherent reliability and positive-rate shifts;
    - source-specific degradation;
    - source-specific positive-rate bias.

    Rect-ROSE later combines source projections arbitrarily.
    """
    regimes: list[
        tuple[np.ndarray, np.ndarray, str]
    ] = []

    n_sources = len(sources)
    nominal_alpha = np.array(
        [source.alpha for source in sources],
        dtype=float,
    )
    nominal_beta = np.array(
        [source.beta for source in sources],
        dtype=float,
    )

    regimes.append(
        (
            nominal_alpha.copy(),
            nominal_beta.copy(),
            "nominal",
        )
    )

    for level in ROBUST_LEVELS:
        coherent_shifts = [
            (-level, +level, "degraded"),
            (+level, -level, "optimistic"),
            (-level, -level, "low_rate"),
            (+level, +level, "high_rate"),
        ]

        for delta_alpha, delta_beta, name in coherent_shifts:
            regimes.append(
                (
                    np.clip(
                        nominal_alpha + delta_alpha,
                        1e-6,
                        1.0 - 1e-6,
                    ),
                    np.clip(
                        nominal_beta + delta_beta,
                        1e-6,
                        1.0 - 1e-6,
                    ),
                    f"{name}_{level:.2f}",
                )
            )

    for source_index in range(n_sources):
        for level in [0.10, 0.20]:
            alpha = nominal_alpha.copy()
            beta = nominal_beta.copy()

            alpha[source_index] = np.clip(
                alpha[source_index] - level,
                1e-6,
                1.0 - 1e-6,
            )
            beta[source_index] = np.clip(
                beta[source_index] + level,
                1e-6,
                1.0 - 1e-6,
            )

            regimes.append(
                (
                    alpha,
                    beta,
                    (
                        f"source_{source_index + 1}_"
                        f"degraded_{level:.2f}"
                    ),
                )
            )

        alpha = nominal_alpha.copy()
        beta = nominal_beta.copy()

        alpha[source_index] = np.clip(
            alpha[source_index] + 0.10,
            1e-6,
            1.0 - 1e-6,
        )
        beta[source_index] = np.clip(
            beta[source_index] + 0.20,
            1e-6,
            1.0 - 1e-6,
        )

        regimes.append(
            (
                alpha,
                beta,
                f"source_{source_index + 1}_positive_bias",
            )
        )

    scenario_channels: list[np.ndarray] = []
    scenario_names: list[str] = []
    seen = set()

    for alpha_vector, beta_vector, name in regimes:
        channels = np.stack(
            [
                make_channel(
                    float(alpha),
                    float(beta),
                )
                for alpha, beta in zip(
                    alpha_vector,
                    beta_vector,
                )
            ]
        )

        key = tuple(
            np.round(channels.ravel(), 12)
        )

        if key in seen:
            continue

        seen.add(key)
        scenario_channels.append(channels)
        scenario_names.append(name)

    scenarios = np.stack(scenario_channels)

    source_projection_points: list[
        list[tuple[float, float]]
    ] = []

    for source_index in range(n_sources):
        points: list[tuple[float, float]] = []

        for scenario_index in range(
            scenarios.shape[0]
        ):
            channel = scenarios[
                scenario_index,
                source_index,
            ]
            point = (
                float(channel[1, 1]),
                float(channel[0, 1]),
            )

            if not any(
                np.allclose(point, existing)
                for existing in points
            ):
                points.append(point)

        source_projection_points.append(points)

    return (
        scenarios,
        source_projection_points,
        scenario_names,
    )



# SHIFT CONSTRUCTION


def mix_channel_collections(
    nominal_channels: np.ndarray,
    target_channels: np.ndarray,
    xi: float,
) -> np.ndarray:
    result = (
        (1.0 - xi) * nominal_channels
        + xi * target_channels
    )

    # Mixtures of valid row-stochastic channels remain valid, but
    # normalization is applied for numerical safety.
    result = result / result.sum(
        axis=-1,
        keepdims=True,
    )
    return result


def target_channels_for_shift(
    nominal_channels: np.ndarray,
    costs: np.ndarray,
    shift_type: str,
) -> np.ndarray:
    n_sources = nominal_channels.shape[0]
    target = nominal_channels.copy()

    if shift_type == "degradation":
        for source_index in range(n_sources):
            target[source_index] = make_channel(
                alpha=0.55,
                beta=0.45,
            )

    elif shift_type == "asymmetric_bias":
        for source_index in range(n_sources):
            target[source_index] = make_channel(
                alpha=0.95,
                beta=0.78,
            )

    elif shift_type == "adversarial_source":
        n_attacked = max(
            1,
            int(
                math.ceil(
                    ADVERSARIAL_SOURCE_FRACTION
                    * n_sources
                )
            ),
        )

        # Attack the cheapest sources, which are likely to be selected
        # by a nominal risk-cost policy.
        attacked_indices = np.argsort(
            costs
        )[:n_attacked]

        for source_index in attacked_indices:
            target[source_index] = make_channel(
                alpha=0.10,
                beta=0.90,
            )

    elif shift_type == "common_mode_failure":
        for source_index in range(n_sources):
            target[source_index] = make_channel(
                alpha=0.52,
                beta=0.48,
            )

    else:
        raise ValueError(
            f"Unknown shift type: {shift_type}"
        )

    return target


def true_regimes_for_shift(
    nominal_channels: np.ndarray,
    costs: np.ndarray,
    shift_type: str,
    xi: float,
) -> tuple[list[np.ndarray], np.ndarray]:
    """
    Return the fixed true regimes and their episode-level probabilities.

    The first three shift types produce one fixed shifted channel model.

    common_mode_failure uses an episode-level latent regime:
    - nominal regime with probability 1-xi;
    - joint failure regime with probability xi.

    This creates cross-source dependence marginally across episodes.
    """
    target = target_channels_for_shift(
        nominal_channels,
        costs,
        shift_type,
    )

    if shift_type == "common_mode_failure":
        if xi <= 0.0:
            return [nominal_channels], np.array([1.0])

        if xi >= 1.0:
            return [target], np.array([1.0])

        return (
            [nominal_channels, target],
            np.array([1.0 - xi, xi]),
        )

    shifted = mix_channel_collections(
        nominal_channels,
        target,
        xi,
    )
    return [shifted], np.array([1.0])



# STATIC-ROSE POLICY MIXTURE


def solve_restricted_policy_mixture(
    policy_risk_matrix: np.ndarray,
) -> tuple[np.ndarray, float]:
    """
    Solve the minimizer's restricted primal game.

    policy_risk_matrix[p,s] is the risk of deterministic policy p
    under ambiguity scenario s.
    """
    n_policies, n_scenarios = (
        policy_risk_matrix.shape
    )

    # Variables are policy weights w_1,...,w_P and scalar t.
    objective = np.zeros(n_policies + 1)
    objective[-1] = 1.0

    inequality_matrix = np.zeros(
        (n_scenarios, n_policies + 1)
    )
    inequality_matrix[:, :n_policies] = (
        policy_risk_matrix.T
    )
    inequality_matrix[:, -1] = -1.0
    inequality_rhs = np.zeros(n_scenarios)

    equality_matrix = np.zeros(
        (1, n_policies + 1)
    )
    equality_matrix[0, :n_policies] = 1.0
    equality_rhs = np.array([1.0])

    bounds = (
        [(0.0, None)] * n_policies
        + [(None, None)]
    )

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            "Static policy-mixture LP failed: "
            f"{result.message}"
        )

    weights = np.asarray(result.x[:n_policies], dtype=float).copy()
    if not np.all(np.isfinite(weights)) or np.min(weights) < -1e-8:
        raise RuntimeError("Policy-mixture LP returned invalid policy weights.")
    # Repair only negative solver roundoff; retain every positive policy mass.
    weights = np.maximum(weights, 0.0)
    mass = float(weights.sum())
    if mass <= 0.0 or abs(mass - 1.0) > 1e-8:
        raise RuntimeError("Policy-mixture weights do not sum to one within tolerance.")
    weights /= mass
    value = float(np.max(policy_risk_matrix.T @ weights))
    if abs(value - float(result.x[-1])) > 1e-8:
        raise RuntimeError("Implemented policy mixture disagrees with the LP objective.")
    return weights, value


def build_static_rose_policy(
    prior_y1: float,
    robust_scenarios: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
) -> dict[str, Any]:
    (
        static_value,
        adversary_distribution,
        policy_risk_matrix,
        policy_trees,
        convergence_history,
    ) = static_double_oracle(
        prior_y1,
        robust_scenarios,
        costs,
        abstention_cost,
        tolerance=STATIC_SOLVER_TOL,
    )

    policy_weights, restricted_value = (
        solve_restricted_policy_mixture(
            policy_risk_matrix
        )
    )

    if not np.isclose(
        static_value,
        restricted_value,
        atol=1e-7,
    ):
        raise AssertionError(
            "Static primal and dual values disagree: "
            f"{static_value} vs {restricted_value}."
        )

    return {
        "value": static_value,
        "policy_weights": policy_weights,
        "policy_trees": policy_trees,
        "adversary_distribution": (
            adversary_distribution
        ),
        "convergence_history": (
            convergence_history
        ),
    }



# EXACT POLICY EVALUATION UNDER A FIXED TRUE MODEL


def evaluate_policy_tree_under_true_model(
    policy_tree: tuple,
    prior_y1: float,
    true_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
) -> dict[str, float]:
    def recurse(
        tree: tuple,
        true_omega: np.ndarray,
    ) -> dict[str, float]:
        action, children = tree
        total = float(true_omega.sum())
        true_p_y1 = float(
            true_omega[1] / total
        )

        if not action.startswith("query_"):
            return terminal_metrics(
                action,
                true_p_y1,
                abstention_cost,
            )

        source_index = int(action.split("_")[1])

        child_items = []

        for evidence in (0, 1):
            next_true_omega = (
                true_omega
                * true_channels[
                    source_index, :, evidence
                ]
            )
            evidence_probability = float(
                next_true_omega.sum() / total
            )

            if evidence_probability > 0.0:
                child_metrics = recurse(
                    children[evidence],
                    next_true_omega,
                )
                child_items.append(
                    (
                        evidence_probability,
                        child_metrics,
                    )
                )

        return add_query_to_metrics(
            weighted_metric_sum(child_items),
            costs[source_index],
        )

    initial_omega = np.array(
        [1.0 - prior_y1, prior_y1],
        dtype=float,
    )

    return recurse(
        policy_tree,
        initial_omega,
    )



def evaluate_nominal_memo_under_true_model(
    prior_y1: float,
    estimated_channels: np.ndarray,
    true_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    nominal_memo: dict | None = None,
) -> dict[str, float]:
    """
    Evaluate the plug-in nominal policy under a fixed true source model.

    The earlier implementation looked up every reached state in the memo
    created by ``nominal_dp``. Because that memo used rounded
    *unnormalized* beliefs as dictionary keys, mathematically identical
    terminal beliefs could differ at the last decimal place when sources
    were multiplied in a different floating-point order. This produced a
    rare KeyError in long experiments.

    This implementation reconstructs the nominal optimal action locally
    from the estimated state. It therefore implements the same Bellman
    policy without relying on exact floating-point memo-key equality.
    ``nominal_memo`` is retained only for backward-compatible calls.
    """
    n_sources = len(costs)
    action_cache: dict = {}

    def estimated_value_and_action(
        estimated_omega: np.ndarray,
        remaining: tuple[int, ...],
    ) -> tuple[float, str]:
        total = float(estimated_omega.sum())

        if total <= 0.0:
            raise ValueError(
                "Estimated belief has zero total probability."
            )

        # A normalized posterior is a stable state representation.
        estimated_p_y1 = float(
            estimated_omega[1] / total
        )
        cache_key = (
            round(estimated_p_y1, 13),
            remaining,
        )

        if cache_key in action_cache:
            return action_cache[cache_key]

        best_value, best_action = stop_value_precise(
            estimated_p_y1,
            abstention_cost,
        )

        for source_index in remaining:
            next_remaining = tuple(
                index
                for index in remaining
                if index != source_index
            )

            continuation_value = float(
                costs[source_index]
            )

            for evidence in (0, 1):
                next_estimated_omega = (
                    estimated_omega
                    * estimated_channels[
                        source_index, :, evidence
                    ]
                )

                evidence_probability = float(
                    next_estimated_omega.sum() / total
                )

                if evidence_probability > 0.0:
                    child_value, _ = (
                        estimated_value_and_action(
                            next_estimated_omega,
                            next_remaining,
                        )
                    )
                    continuation_value += (
                        evidence_probability
                        * child_value
                    )

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        action_cache[cache_key] = (
            float(best_value),
            str(best_action),
        )
        return action_cache[cache_key]

    def evaluate_true_metrics(
        estimated_omega: np.ndarray,
        true_omega: np.ndarray,
        remaining: tuple[int, ...],
    ) -> dict[str, float]:
        _, action = estimated_value_and_action(
            estimated_omega,
            remaining,
        )

        true_total = float(true_omega.sum())

        if true_total <= 0.0:
            raise ValueError(
                "True belief has zero total probability."
            )

        true_p_y1 = float(
            true_omega[1] / true_total
        )

        if not action.startswith("query_"):
            return terminal_metrics(
                action,
                true_p_y1,
                abstention_cost,
            )

        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            index
            for index in remaining
            if index != source_index
        )

        child_items: list[
            tuple[float, dict[str, float]]
        ] = []

        for evidence in (0, 1):
            next_estimated_omega = (
                estimated_omega
                * estimated_channels[
                    source_index, :, evidence
                ]
            )
            next_true_omega = (
                true_omega
                * true_channels[
                    source_index, :, evidence
                ]
            )

            true_evidence_probability = float(
                next_true_omega.sum() / true_total
            )

            if true_evidence_probability <= 0.0:
                continue

            # In this experiment all estimated channels are strictly
            # interior. The guard makes the evaluator explicit and safe
            # if a future extension introduces exact 0/1 estimates.
            if float(next_estimated_omega.sum()) <= 0.0:
                estimated_p_y1 = float(
                    estimated_omega[1]
                    / estimated_omega.sum()
                )
                next_estimated_omega = np.array(
                    [
                        1.0 - estimated_p_y1,
                        estimated_p_y1,
                    ],
                    dtype=float,
                )

            child_items.append(
                (
                    true_evidence_probability,
                    evaluate_true_metrics(
                        next_estimated_omega,
                        next_true_omega,
                        next_remaining,
                    ),
                )
            )

        return add_query_to_metrics(
            weighted_metric_sum(child_items),
            costs[source_index],
        )

    initial_omega = np.array(
        [1.0 - prior_y1, prior_y1],
        dtype=float,
    )

    return evaluate_true_metrics(
        initial_omega.copy(),
        initial_omega.copy(),
        tuple(range(n_sources)),
    )



def evaluate_hcr_memo_under_true_model(
    prior_y1: float,
    true_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    hcr_memo: dict,
) -> dict[str, float]:
    def recurse(
        history: tuple[tuple[int, int], ...],
        remaining: tuple[int, ...],
        true_omega: np.ndarray,
    ) -> dict[str, float]:
        node = hcr_memo[(history, remaining)]
        action = node[1]

        true_total = float(true_omega.sum())
        true_p_y1 = float(
            true_omega[1] / true_total
        )

        if not action.startswith("query_"):
            return terminal_metrics(
                action,
                true_p_y1,
                abstention_cost,
            )

        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            index
            for index in remaining
            if index != source_index
        )

        child_items = []

        for evidence in (0, 1):
            next_true_omega = (
                true_omega
                * true_channels[
                    source_index, :, evidence
                ]
            )
            evidence_probability = float(
                next_true_omega.sum() / true_total
            )

            if evidence_probability > 0.0:
                child_items.append(
                    (
                        evidence_probability,
                        recurse(
                            history
                            + ((source_index, evidence),),
                            next_remaining,
                            next_true_omega,
                        ),
                    )
                )

        return add_query_to_metrics(
            weighted_metric_sum(child_items),
            costs[source_index],
        )

    initial_omega = np.array(
        [1.0 - prior_y1, prior_y1],
        dtype=float,
    )

    return recurse(
        tuple(),
        tuple(range(len(costs))),
        initial_omega,
    )


def rectangular_child_belief_set(
    belief_set: tuple[float, ...],
    source_channels: list[np.ndarray],
    evidence: int,
) -> tuple[float, ...]:
    posteriors: list[float] = []

    for belief in belief_set:
        for channel in source_channels:
            denominator = (
                (1.0 - belief)
                * channel[0, evidence]
                + belief
                * channel[1, evidence]
            )

            if denominator > 0.0:
                posterior = (
                    belief
                    * channel[1, evidence]
                    / denominator
                )
                posteriors.append(
                    float(posterior)
                )

    return _deduplicate_beliefs(posteriors)


def evaluate_rect_memo_under_true_model(
    prior_y1: float,
    true_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    source_projection_points: list[
        list[tuple[float, float]]
    ],
    rect_memo: dict,
) -> dict[str, float]:
    channel_sets = [
        [
            make_channel(alpha, beta)
            for alpha, beta in source_points
        ]
        for source_points in source_projection_points
    ]

    def recurse(
        belief_set: tuple[float, ...],
        remaining: tuple[int, ...],
        true_omega: np.ndarray,
    ) -> dict[str, float]:
        belief_set = _deduplicate_beliefs(
            belief_set
        )
        node = rect_memo[
            (belief_set, remaining)
        ]
        action = node[1]

        true_total = float(true_omega.sum())
        true_p_y1 = float(
            true_omega[1] / true_total
        )

        if not action.startswith("query_"):
            return terminal_metrics(
                action,
                true_p_y1,
                abstention_cost,
            )

        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            index
            for index in remaining
            if index != source_index
        )

        child_items = []

        for evidence in (0, 1):
            next_true_omega = (
                true_omega
                * true_channels[
                    source_index, :, evidence
                ]
            )
            evidence_probability = float(
                next_true_omega.sum() / true_total
            )

            if evidence_probability > 0.0:
                child_beliefs = (
                    rectangular_child_belief_set(
                        belief_set,
                        channel_sets[source_index],
                        evidence,
                    )
                )

                child_items.append(
                    (
                        evidence_probability,
                        recurse(
                            child_beliefs,
                            next_remaining,
                            next_true_omega,
                        ),
                    )
                )

        return add_query_to_metrics(
            weighted_metric_sum(child_items),
            costs[source_index],
        )

    initial_omega = np.array(
        [1.0 - prior_y1, prior_y1],
        dtype=float,
    )

    return recurse(
        (prior_y1,),
        tuple(range(len(costs))),
        initial_omega,
    )


def evaluate_retrieve_all_under_true_model(
    prior_y1: float,
    nominal_channels: np.ndarray,
    true_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
) -> dict[str, float]:
    query_order = tuple(
        np.argsort(costs).tolist()
    )

    def recurse(
        position: int,
        nominal_omega: np.ndarray,
        true_omega: np.ndarray,
    ) -> dict[str, float]:
        true_total = float(true_omega.sum())

        if position == len(query_order):
            nominal_p_y1 = float(
                nominal_omega[1]
                / nominal_omega.sum()
            )
            _, terminal_action = (
                stop_value_precise(
                    nominal_p_y1,
                    abstention_cost,
                )
            )
            true_p_y1 = float(
                true_omega[1] / true_total
            )
            return terminal_metrics(
                terminal_action,
                true_p_y1,
                abstention_cost,
            )

        source_index = query_order[position]
        child_items = []

        for evidence in (0, 1):
            next_nominal_omega = (
                nominal_omega
                * nominal_channels[
                    source_index, :, evidence
                ]
            )
            next_true_omega = (
                true_omega
                * true_channels[
                    source_index, :, evidence
                ]
            )
            evidence_probability = float(
                next_true_omega.sum() / true_total
            )

            if evidence_probability > 0.0:
                child_items.append(
                    (
                        evidence_probability,
                        recurse(
                            position + 1,
                            next_nominal_omega,
                            next_true_omega,
                        ),
                    )
                )

        return add_query_to_metrics(
            weighted_metric_sum(child_items),
            costs[source_index],
        )

    initial_omega = np.array(
        [1.0 - prior_y1, prior_y1],
        dtype=float,
    )

    return recurse(
        0,
        initial_omega.copy(),
        initial_omega.copy(),
    )



# PRECOMPUTE ALL POLICIES FOR ONE NOMINAL INSTANCE


def build_instance_policies(
    instance_id: int,
) -> dict[str, Any]:
    problem = generate_nominal_problem(
        instance_id
    )

    sources = problem["sources"]
    prior_y1 = problem["prior_y1"]
    abstention_cost = problem["abstention_cost"]
    nominal_channels = problem[
        "nominal_channels"
    ]
    costs = problem["costs"]

    (
        robust_scenarios,
        source_projection_points,
        scenario_names,
    ) = build_robust_scenario_family(sources)

    nominal_root, nominal_memo = nominal_dp(
        prior_y1,
        nominal_channels,
        costs,
        abstention_cost,
    )

    static_policy = build_static_rose_policy(
        prior_y1,
        robust_scenarios,
        costs,
        abstention_cost,
    )

    hcr_root, hcr_memo = hcr_dp(
        prior_y1,
        robust_scenarios,
        costs,
        abstention_cost,
    )

    rect_root, rect_memo = rectangular_dp(
        prior_y1,
        source_projection_points,
        costs,
        abstention_cost,
    )

    internal_value, internal_action = (
        stop_value_precise(
            prior_y1,
            abstention_cost,
        )
    )

    static_root_actions = [
        tree[0]
        for tree in static_policy[
            "policy_trees"
        ]
    ]

    return {
        **problem,
        "robust_scenarios": robust_scenarios,
        "source_projection_points": (
            source_projection_points
        ),
        "scenario_names": scenario_names,
        "nominal_root": nominal_root,
        "nominal_memo": nominal_memo,
        "static_policy": static_policy,
        "hcr_root": hcr_root,
        "hcr_memo": hcr_memo,
        "rect_root": rect_root,
        "rect_memo": rect_memo,
        "internal_action": internal_action,
        "internal_value": internal_value,
        "static_root_actions": (
            static_root_actions
        ),
    }



# EVALUATE ALL METHODS FOR ONE TRUE REGIME MIXTURE


def evaluate_method_under_regimes(
    method: str,
    policies: dict[str, Any],
    true_regimes: list[np.ndarray],
    regime_weights: np.ndarray,
) -> tuple[
    dict[str, float],
    dict[str, float],
    dict[str, Any],
]:
    prior_y1 = policies["prior_y1"]
    abstention_cost = policies[
        "abstention_cost"
    ]
    nominal_channels = policies[
        "nominal_channels"
    ]
    costs = policies["costs"]

    regime_metrics: list[
        dict[str, float]
    ] = []
    regime_root_actions: list[str] = []
    extra: dict[str, Any] = {}

    for true_channels in true_regimes:
        if method == "InternalOnly":
            metrics = terminal_metrics(
                policies["internal_action"],
                prior_y1,
                abstention_cost,
            )
            root_action = policies[
                "internal_action"
            ]

        elif method == "RetrieveAll":
            metrics = (
                evaluate_retrieve_all_under_true_model(
                    prior_y1,
                    nominal_channels,
                    true_channels,
                    costs,
                    abstention_cost,
                )
            )
            root_action = (
                f"query_{int(np.argsort(costs)[0])}"
            )

        elif method == "NominalSequential":
            metrics = (
                evaluate_nominal_memo_under_true_model(
                    prior_y1,
                    nominal_channels,
                    true_channels,
                    costs,
                    abstention_cost,
                    policies["nominal_memo"],
                )
            )
            root_action = policies[
                "nominal_root"
            ][1]

        elif method == "StaticROSE":
            tree_metrics = [
                evaluate_policy_tree_under_true_model(
                    tree,
                    prior_y1,
                    true_channels,
                    costs,
                    abstention_cost,
                )
                for tree in policies[
                    "static_policy"
                ]["policy_trees"]
            ]

            metrics = weighted_metric_sum(
                list(
                    zip(
                        policies[
                            "static_policy"
                        ]["policy_weights"],
                        tree_metrics,
                    )
                )
            )
            root_action = "randomized_static"

        elif method == "HCRROSE":
            metrics = (
                evaluate_hcr_memo_under_true_model(
                    prior_y1,
                    true_channels,
                    costs,
                    abstention_cost,
                    policies["hcr_memo"],
                )
            )
            root_action = policies[
                "hcr_root"
            ][1]

        elif method == "RectROSE":
            metrics = (
                evaluate_rect_memo_under_true_model(
                    prior_y1,
                    true_channels,
                    costs,
                    abstention_cost,
                    policies[
                        "source_projection_points"
                    ],
                    policies["rect_memo"],
                )
            )
            root_action = policies[
                "rect_root"
            ][1]

        elif method == "Oracle":
            oracle_root, oracle_memo = nominal_dp(
                prior_y1,
                true_channels,
                costs,
                abstention_cost,
            )

            metrics = (
                evaluate_nominal_memo_under_true_model(
                    prior_y1,
                    true_channels,
                    true_channels,
                    costs,
                    abstention_cost,
                    oracle_memo,
                )
            )
            root_action = oracle_root[1]

        else:
            raise ValueError(
                f"Unknown method: {method}"
            )

        regime_metrics.append(metrics)
        regime_root_actions.append(
            root_action
        )

    combined_metrics = combine_true_regimes(
        regime_metrics,
        regime_weights,
    )

    if method == "StaticROSE":
        root_probabilities = (
            root_action_probabilities(
                policies["static_root_actions"],
                policies[
                    "static_policy"
                ]["policy_weights"],
            )
        )
    else:
        root_probabilities = (
            root_action_probabilities(
                regime_root_actions,
                regime_weights,
            )
        )

    if method == "Oracle" and len(
        true_regimes
    ) > 1:
        extra[
            "oracle_common_mode_definition"
        ] = "clairvoyant_regime_lower_bound"
    else:
        extra[
            "oracle_common_mode_definition"
        ] = "exact_fixed_model_oracle"

    return (
        combined_metrics,
        root_probabilities,
        extra,
    )



# ONE INSTANCE


def solve_misspecification_instance(
    instance_id: int,
) -> pd.DataFrame:
    policy_start = time.perf_counter()
    policies = build_instance_policies(
        instance_id
    )
    policy_runtime = (
        time.perf_counter() - policy_start
    )

    rows: list[dict[str, Any]] = []

    for shift_type in SHIFT_TYPES:
        for xi in SHIFT_LEVELS:
            (
                true_regimes,
                regime_weights,
            ) = true_regimes_for_shift(
                policies["nominal_channels"],
                policies["costs"],
                shift_type,
                float(xi),
            )

            method_results: dict[
                str, dict[str, float]
            ] = {}

            method_root_results: dict[
                str, dict[str, float]
            ] = {}

            method_extras: dict[
                str, dict[str, Any]
            ] = {}

            for method in METHODS:
                (
                    metrics,
                    root_probabilities,
                    extra,
                ) = evaluate_method_under_regimes(
                    method,
                    policies,
                    true_regimes,
                    regime_weights,
                )

                method_results[method] = metrics
                method_root_results[
                    method
                ] = root_probabilities
                method_extras[method] = extra

            oracle_risk = method_results[
                "Oracle"
            ]["total_risk"]

            for method in METHODS:
                metrics = method_results[method]
                root_probabilities = (
                    method_root_results[method]
                )

                row: dict[str, Any] = {
                    "instance_id": instance_id,
                    "shift_type": shift_type,
                    "shift_level": float(xi),
                    "method": method,
                    "prior_y1": policies[
                        "prior_y1"
                    ],
                    "abstention_cost": policies[
                        "abstention_cost"
                    ],
                    "n_sources": N_SOURCES,
                    "n_robust_scenarios": int(
                        policies[
                            "robust_scenarios"
                        ].shape[0]
                    ),
                    "policy_build_runtime_sec": (
                        policy_runtime
                    ),
                    "true_regime_count": len(
                        true_regimes
                    ),
                    "oracle_risk": oracle_risk,
                    "oracle_regret": (
                        metrics["total_risk"]
                        - oracle_risk
                    ),
                    **metrics,
                    **root_probabilities,
                    **method_extras[method],
                }

                for source_index, source in enumerate(
                    policies["sources"]
                ):
                    source_number = (
                        source_index + 1
                    )
                    row[
                        f"source_{source_number}_alpha"
                    ] = source.alpha
                    row[
                        f"source_{source_number}_beta"
                    ] = source.beta
                    row[
                        f"source_{source_number}_cost"
                    ] = source.cost

                rows.append(row)

    result = pd.DataFrame(rows)

    # Exact metric identity check.
    reconstructed_risk = (
        result["wrong_answer_probability"]
        + result["abstention_cost"]
        * result["abstention_probability"]
        + result["acquisition_cost"]
    )

    if not np.allclose(
        result["total_risk"],
        reconstructed_risk,
        atol=RISK_TOL,
    ):
        raise AssertionError(
            "Metric decomposition does not reconstruct total risk."
        )

    return result



# SUMMARY TABLES


def normal_mean_interval(
    values: pd.Series,
) -> tuple[float, float, float]:
    values = values.dropna().to_numpy(
        dtype=float
    )
    n = len(values)

    if n == 0:
        return np.nan, np.nan, np.nan

    mean = float(np.mean(values))

    if n == 1:
        return mean, np.nan, np.nan

    standard_error = float(
        np.std(values, ddof=1)
        / math.sqrt(n)
    )

    lower = mean - 1.959963984540054 * standard_error
    upper = mean + 1.959963984540054 * standard_error
    return mean, lower, upper


def summarize_misspecification(
    results_df: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    summary_rows: list[dict[str, Any]] = []

    group_columns = [
        "shift_type",
        "shift_level",
        "method",
    ]

    for group_key, group in results_df.groupby(
        group_columns
    ):
        (
            shift_type,
            shift_level,
            method,
        ) = group_key

        (
            mean_risk,
            risk_ci_lower,
            risk_ci_upper,
        ) = normal_mean_interval(
            group["total_risk"]
        )

        (
            mean_regret,
            regret_ci_lower,
            regret_ci_upper,
        ) = normal_mean_interval(
            group["oracle_regret"]
        )

        summary_rows.append(
            {
                "shift_type": shift_type,
                "shift_level": float(
                    shift_level
                ),
                "method": method,
                "n_instances": len(group),
                "mean_true_risk": mean_risk,
                "risk_ci_lower": risk_ci_lower,
                "risk_ci_upper": risk_ci_upper,
                "median_true_risk": float(
                    group[
                        "total_risk"
                    ].median()
                ),
                "mean_oracle_regret": (
                    mean_regret
                ),
                "regret_ci_lower": (
                    regret_ci_lower
                ),
                "regret_ci_upper": (
                    regret_ci_upper
                ),
                "mean_wrong_answer_probability": float(
                    group[
                        "wrong_answer_probability"
                    ].mean()
                ),
                "mean_abstention_probability": float(
                    group[
                        "abstention_probability"
                    ].mean()
                ),
                "mean_answer_probability": float(
                    group[
                        "answer_probability"
                    ].mean()
                ),
                "mean_acquisition_cost": float(
                    group[
                        "acquisition_cost"
                    ].mean()
                ),
                "mean_query_count": float(
                    group[
                        "expected_query_count"
                    ].mean()
                ),
                "mean_root_query_probability": float(
                    group[
                        "root_query_probability"
                    ].mean()
                ),
                "mean_root_answer_probability": float(
                    group[
                        "root_answer_probability"
                    ].mean()
                ),
                "mean_root_abstain_probability": float(
                    group[
                        "root_abstain_probability"
                    ].mean()
                ),
            }
        )

    summary_df = pd.DataFrame(
        summary_rows
    ).sort_values(group_columns)

    worst_case_df = (
        results_df.groupby(
            ["instance_id", "method"],
            as_index=False,
        )["total_risk"]
        .max()
        .rename(
            columns={
                "total_risk": "instance_worst_case_risk"
            }
        )
    )

    worst_case_summary_rows = []

    for method, group in worst_case_df.groupby(
        "method"
    ):
        (
            mean_value,
            lower,
            upper,
        ) = normal_mean_interval(
            group["instance_worst_case_risk"]
        )

        worst_case_summary_rows.append(
            {
                "method": method,
                "mean_worst_case_risk": mean_value,
                "ci_lower": lower,
                "ci_upper": upper,
                "median_worst_case_risk": float(
                    group[
                        "instance_worst_case_risk"
                    ].median()
                ),
            }
        )

    worst_case_summary_df = pd.DataFrame(
        worst_case_summary_rows
    ).sort_values(
        "mean_worst_case_risk"
    )

    action_summary_df = summary_df[
        [
            "shift_type",
            "shift_level",
            "method",
            "mean_root_query_probability",
            "mean_root_answer_probability",
            "mean_root_abstain_probability",
            "mean_query_count",
            "mean_acquisition_cost",
        ]
    ].copy()

    shift_type_summary_df = (
        summary_df.groupby(
            ["shift_type", "method"],
            as_index=False,
        )
        .agg(
            mean_risk_across_levels=(
                "mean_true_risk",
                "mean",
            ),
            maximum_mean_risk=(
                "mean_true_risk",
                "max",
            ),
            mean_oracle_regret_across_levels=(
                "mean_oracle_regret",
                "mean",
            ),
        )
    )

    return (
        summary_df,
        worst_case_summary_df,
        action_summary_df,
        shift_type_summary_df,
    )



# VECTOR FIGURES


PLOT_METHODS = [
    "InternalOnly",
    "RetrieveAll",
    "NominalSequential",
    "StaticROSE",
    "HCRROSE",
    "RectROSE",
    "Oracle",
]


def plot_risk_by_shift_type(
    summary_df: pd.DataFrame,
    shift_type: str,
) -> None:
    subset = summary_df[
        summary_df["shift_type"]
        == shift_type
    ]

    figure = plt.figure(
        figsize=(9.0, 5.4)
    )

    for method in PLOT_METHODS:
        group = (
            subset[
                subset["method"] == method
            ]
            .sort_values("shift_level")
        )

        if group.empty:
            continue

        plt.plot(
            group["shift_level"],
            group["mean_true_risk"],
            marker="o",
            label=method,
        )

    plt.xlabel("Misspecification intensity")
    plt.ylabel("Mean true decision risk")
    plt.title(
        "Decision risk under "
        + shift_type.replace("_", " ")
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        f"figure_risk_vs_shift_{shift_type}",
    )
    plt.close(figure)


def plot_worst_case_risk(
    worst_case_summary_df: pd.DataFrame,
) -> None:
    ordered = (
        worst_case_summary_df
        .sort_values(
            "mean_worst_case_risk"
        )
        .reset_index(drop=True)
    )

    x = np.arange(len(ordered))
    values = ordered[
        "mean_worst_case_risk"
    ].to_numpy()

    lower = (
        values
        - ordered["ci_lower"].to_numpy()
    )
    upper = (
        ordered["ci_upper"].to_numpy()
        - values
    )

    figure = plt.figure(
        figsize=(9.0, 5.4)
    )
    plt.bar(
        x,
        values,
    )
    plt.errorbar(
        x,
        values,
        yerr=np.vstack([lower, upper]),
        fmt="none",
        capsize=4,
    )
    plt.xticks(
        x,
        ordered["method"],
        rotation=25,
        ha="right",
    )
    plt.ylabel(
        "Mean instance-level worst-case risk"
    )
    plt.title(
        "Worst-case risk across shift types and intensities"
    )
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_worst_case_risk",
    )
    plt.close(figure)


def aggregate_across_shift_types(
    summary_df: pd.DataFrame,
) -> pd.DataFrame:
    return (
        summary_df.groupby(
            ["shift_level", "method"],
            as_index=False,
        )
        .agg(
            mean_true_risk=(
                "mean_true_risk",
                "mean",
            ),
            mean_oracle_regret=(
                "mean_oracle_regret",
                "mean",
            ),
            mean_abstention_probability=(
                "mean_abstention_probability",
                "mean",
            ),
            mean_acquisition_cost=(
                "mean_acquisition_cost",
                "mean",
            ),
            mean_root_query_probability=(
                "mean_root_query_probability",
                "mean",
            ),
        )
    )


def plot_oracle_regret(
    aggregated_df: pd.DataFrame,
) -> None:
    figure = plt.figure(
        figsize=(9.0, 5.4)
    )

    for method in PLOT_METHODS:
        if method == "Oracle":
            continue

        group = (
            aggregated_df[
                aggregated_df["method"]
                == method
            ]
            .sort_values("shift_level")
        )

        plt.plot(
            group["shift_level"],
            group["mean_oracle_regret"],
            marker="o",
            label=method,
        )

    plt.axhline(0.0, linestyle="--")
    plt.xlabel("Misspecification intensity")
    plt.ylabel("Mean regret to oracle")
    plt.title(
        "Oracle regret averaged across shift types"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_oracle_regret_vs_shift",
    )
    plt.close(figure)


def plot_abstention(
    aggregated_df: pd.DataFrame,
) -> None:
    figure = plt.figure(
        figsize=(9.0, 5.4)
    )

    for method in PLOT_METHODS:
        group = (
            aggregated_df[
                aggregated_df["method"]
                == method
            ]
            .sort_values("shift_level")
        )

        plt.plot(
            group["shift_level"],
            group[
                "mean_abstention_probability"
            ],
            marker="o",
            label=method,
        )

    plt.xlabel("Misspecification intensity")
    plt.ylabel("Mean abstention probability")
    plt.title(
        "Abstention under source-model misspecification"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_abstention_vs_shift",
    )
    plt.close(figure)


def plot_acquisition_cost(
    aggregated_df: pd.DataFrame,
) -> None:
    figure = plt.figure(
        figsize=(9.0, 5.4)
    )

    for method in PLOT_METHODS:
        group = (
            aggregated_df[
                aggregated_df["method"]
                == method
            ]
            .sort_values("shift_level")
        )

        plt.plot(
            group["shift_level"],
            group[
                "mean_acquisition_cost"
            ],
            marker="o",
            label=method,
        )

    plt.xlabel("Misspecification intensity")
    plt.ylabel("Mean acquisition cost")
    plt.title(
        "Retrieval cost under misspecification"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_retrieval_cost_vs_shift",
    )
    plt.close(figure)


def plot_shift_type_comparison(
    shift_type_summary_df: pd.DataFrame,
) -> None:
    methods_without_oracle = [
        method
        for method in PLOT_METHODS
        if method != "Oracle"
    ]

    pivot = (
        shift_type_summary_df[
            shift_type_summary_df["method"].isin(
                methods_without_oracle
            )
        ]
        .pivot(
            index="shift_type",
            columns="method",
            values="maximum_mean_risk",
        )
        .reindex(columns=methods_without_oracle)
    )

    x = np.arange(len(pivot.index))
    total_width = 0.82
    bar_width = total_width / len(
        methods_without_oracle
    )

    figure = plt.figure(
        figsize=(10.5, 5.8)
    )

    for method_index, method in enumerate(
        methods_without_oracle
    ):
        positions = (
            x
            - total_width / 2
            + bar_width / 2
            + method_index * bar_width
        )
        plt.bar(
            positions,
            pivot[method].to_numpy(),
            width=bar_width,
            label=method,
        )

    plt.xticks(
        x,
        [
            label.replace("_", " ")
            for label in pivot.index
        ],
        rotation=20,
        ha="right",
    )
    plt.ylabel("Maximum mean true risk")
    plt.title(
        "Method sensitivity across misspecification types"
    )
    plt.legend()
    plt.tight_layout()
    save_vector_figure(
        figure,
        "figure_shift_type_comparison",
    )
    plt.close(figure)



# CHECKPOINT AND FINAL EXPORT


def experiment_signature() -> dict[str, Any]:
    return {
        "seed": SEED,
        "run_mode": RUN_MODE,
        "n_instances": N_INSTANCES,
        "shift_levels": SHIFT_LEVELS,
        "shift_types": SHIFT_TYPES,
        "methods": METHODS,
        "n_sources": N_SOURCES,
        "robust_levels": ROBUST_LEVELS,
        "adversarial_source_fraction": (
            ADVERSARIAL_SOURCE_FRACTION
        ),
        "static_solver_tolerance": (
            STATIC_SOLVER_TOL
        ),
    }


def validate_checkpoint_signature() -> None:
    signature_path = (
        OUTPUT_DIR / "checkpoint_config.json"
    )
    current = experiment_signature()

    if signature_path.exists():
        existing = json.loads(
            signature_path.read_text(
                encoding="utf-8"
            )
        )

        if existing != current:
            raise RuntimeError(
                "Existing checkpoint configuration differs from "
                "the current experiment. Remove the output directory "
                "or restore the original configuration."
            )
    else:
        signature_path.write_text(
            json.dumps(
                current,
                indent=2,
            ),
            encoding="utf-8",
        )


def finalize_misspecification_results(
    results_df: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    results_df = (
        results_df
        .drop_duplicates(
            subset=[
                "instance_id",
                "shift_type",
                "shift_level",
                "method",
            ],
            keep="last",
        )
        .sort_values(
            [
                "instance_id",
                "shift_type",
                "shift_level",
                "method",
            ]
        )
        .reset_index(drop=True)
    )

    (
        summary_df,
        worst_case_df,
        action_df,
        shift_type_df,
    ) = summarize_misspecification(
        results_df
    )

    results_df.to_csv(
        OUTPUT_DIR
        / "results_misspecification_all.csv",
        index=False,
    )
    summary_df.to_csv(
        OUTPUT_DIR
        / "results_misspecification_summary.csv",
        index=False,
    )
    worst_case_df.to_csv(
        OUTPUT_DIR
        / "results_worst_case_risk.csv",
        index=False,
    )
    action_df.to_csv(
        OUTPUT_DIR
        / "results_action_rates.csv",
        index=False,
    )
    shift_type_df.to_csv(
        OUTPUT_DIR
        / "results_shift_type_summary.csv",
        index=False,
    )

    summary_df[
        [
            "shift_type",
            "shift_level",
            "method",
            "mean_oracle_regret",
            "regret_ci_lower",
            "regret_ci_upper",
        ]
    ].to_csv(
        OUTPUT_DIR
        / "results_oracle_regret.csv",
        index=False,
    )

    for shift_type in SHIFT_TYPES:
        plot_risk_by_shift_type(
            summary_df,
            shift_type,
        )

    plot_worst_case_risk(
        worst_case_df
    )

    aggregated_df = (
        aggregate_across_shift_types(
            summary_df
        )
    )
    aggregated_df.to_csv(
        OUTPUT_DIR
        / "results_aggregated_across_shifts.csv",
        index=False,
    )

    plot_oracle_regret(
        aggregated_df
    )
    plot_abstention(
        aggregated_df
    )
    plot_acquisition_cost(
        aggregated_df
    )
    plot_shift_type_comparison(
        shift_type_df
    )

    metadata = experiment_signature()
    metadata["completed_instances"] = int(
        results_df[
            "instance_id"
        ].nunique()
    )
    metadata["completed_rows"] = int(
        len(results_df)
    )

    (
        OUTPUT_DIR / "experiment_metadata.json"
    ).write_text(
        json.dumps(
            metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        "\nMisspecification experiment finalized."
    )
    print("Output directory:", OUTPUT_DIR)
    print()
    print(
        worst_case_df[
            [
                "method",
                "mean_worst_case_risk",
                "median_worst_case_risk",
            ]
        ].to_string(index=False)
    )

    return {
        "results": results_df,
        "summary": summary_df,
        "worst_case": worst_case_df,
        "action_rates": action_df,
        "shift_type_summary": (
            shift_type_df
        ),
        "aggregated": aggregated_df,
    }


def run_misspecification_experiment(
    resume: bool = RESUME,
) -> dict[str, pd.DataFrame]:
    validate_checkpoint_signature()

    checkpoint_path = (
        OUTPUT_DIR
        / "checkpoint_misspecification.csv"
    )

    if (
        resume
        and checkpoint_path.exists()
        and checkpoint_path.stat().st_size > 0
    ):
        checkpoint_df = pd.read_csv(
            checkpoint_path
        )

        checkpoint_df = (
            checkpoint_df
            .drop_duplicates(
                subset=[
                    "instance_id",
                    "shift_type",
                    "shift_level",
                    "method",
                ],
                keep="last",
            )
        )

        completed_instances = set(
            checkpoint_df[
                "instance_id"
            ].astype(int).unique()
        )

        print(
            f"Resuming from "
            f"{len(completed_instances)} "
            "completed instances."
        )
    else:
        completed_instances = set()

        if checkpoint_path.exists():
            checkpoint_path.unlink()

    pending_instances = [
        instance_id
        for instance_id in range(
            N_INSTANCES
        )
        if instance_id
        not in completed_instances
    ]

    start_time = time.perf_counter()
    batch_frames: list[pd.DataFrame] = []

    def flush_batch() -> None:
        nonlocal batch_frames

        if not batch_frames:
            return

        batch_df = pd.concat(
            batch_frames,
            ignore_index=True,
        )

        write_header = (
            not checkpoint_path.exists()
            or checkpoint_path.stat().st_size
            == 0
        )

        batch_df.to_csv(
            checkpoint_path,
            mode="a",
            header=write_header,
            index=False,
        )

        batch_frames = []

    completed_count = len(
        completed_instances
    )

    for instance_id in pending_instances:
        instance_df = (
            solve_misspecification_instance(
                instance_id
            )
        )
        batch_frames.append(instance_df)
        completed_count += 1

        if (
            len(batch_frames)
            >= CHECKPOINT_EVERY
            or completed_count
            == N_INSTANCES
        ):
            flush_batch()

            elapsed = (
                time.perf_counter()
                - start_time
            )
            print(
                f"Completed "
                f"{completed_count}/"
                f"{N_INSTANCES} instances "
                f"(elapsed this session "
                f"{elapsed:.2f} sec)"
            )

    flush_batch()

    if not checkpoint_path.exists():
        raise RuntimeError(
            "Checkpoint file was not created."
        )

    final_df = pd.read_csv(
        checkpoint_path
    )

    expected_rows = (
        N_INSTANCES
        * len(SHIFT_TYPES)
        * len(SHIFT_LEVELS)
        * len(METHODS)
    )

    unique_key_count = len(
        final_df.drop_duplicates(
            subset=[
                "instance_id",
                "shift_type",
                "shift_level",
                "method",
            ]
        )
    )

    if unique_key_count != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} unique rows "
            f"but found {unique_key_count}."
        )

    # Rewrite a clean checkpoint once at the end.
    final_df = (
        final_df
        .drop_duplicates(
            subset=[
                "instance_id",
                "shift_type",
                "shift_level",
                "method",
            ],
            keep="last",
        )
        .sort_values(
            [
                "instance_id",
                "shift_type",
                "shift_level",
                "method",
            ]
        )
    )
    final_df.to_csv(
        checkpoint_path,
        index=False,
    )

    return finalize_misspecification_results(
        final_df
    )


def recover_misspecification_from_checkpoint(
) -> dict[str, pd.DataFrame]:
    checkpoint_path = (
        OUTPUT_DIR
        / "checkpoint_misspecification.csv"
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: "
            f"{checkpoint_path}"
        )

    checkpoint_df = pd.read_csv(
        checkpoint_path
    )

    print(
        f"Recovered {len(checkpoint_df)} rows."
    )

    return finalize_misspecification_results(
        checkpoint_df
    )



## Execute or resume


## Nominal and oracle policy evaluation

Nominal and oracle actions are evaluated through local dynamic programs rather than floating-point belief-key lookups in a saved memo. HCR and Rectangular retain their own recursions.


In [ ]:
experiment = run_misspecification_experiment(
    resume=True
)

## Result summaries


In [ ]:
from IPython.display import display

print("Worst-case risk:")
display(experiment["worst_case"])

print("Aggregated across shift types:")
display(experiment["aggregated"])

print("Detailed summary:")
display(experiment["summary"])

## Saved vector figures


In [ ]:
from IPython.display import SVG, display

figure_names = [
    "figure_risk_vs_shift_degradation.svg",
    "figure_risk_vs_shift_asymmetric_bias.svg",
    "figure_risk_vs_shift_adversarial_source.svg",
    "figure_risk_vs_shift_common_mode_failure.svg",
    "figure_worst_case_risk.svg",
    "figure_oracle_regret_vs_shift.svg",
    "figure_abstention_vs_shift.svg",
    "figure_retrieval_cost_vs_shift.svg",
    "figure_shift_type_comparison.svg",
]

for filename in figure_names:
    path = OUTPUT_DIR / filename
    if path.exists():
        print(filename)
        display(SVG(filename=str(path)))

## Recover completed results

The recovery function rebuilds final summaries and plots from completed saved results.
